In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# Configuration
DATA_PATH = '../data/dirty_heart_disease_renamed.csv'
TARGET = 'heart_disease_target'

In [ ]:
# Data loading and inspection helpers
def load_data(path):
    return pd.read_csv(path)


def show_value_counts(df):
    for col in df.columns:
        print(col)
        print('=====================')
        print(df[col].value_counts())
        print('\n')

In [ ]:
# Data cleaning helpers
def drop_missing_rows(df):
    df = df.dropna()
    return df.reset_index(drop=True)


def replace_question_marks(df):
    return df.replace('?', np.nan)


def clean_age(df):
    df['age'] = df['age'].replace({'54 years': '54'})
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    return df


def remove_age_outliers(df):
    Q1 = df['age'].quantile(0.25)
    Q3 = df['age'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df = df[(df['age'] >= lower_bound) & (df['age'] <= upper_bound)]
    return df.reset_index(drop=True)


def clean_gender(df):
    df['gender'] = df['gender'].replace({
        '1': 'Male',
        1: 'Male',
        'M': 'Male',
        'F': 'Female'
    })
    most_common_gender = df['gender'].mode()[0]
    df['gender'] = df['gender'].replace('unknown', most_common_gender)
    return df


def clean_cholesterol(df):
    df['cholesterol'] = pd.to_numeric(df['cholesterol'], errors='coerce')
    df.loc[df['cholesterol'] < 0, 'cholesterol'] = np.nan
    df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())
    return df


def clean_resting_blood_pressure(df):
    df['resting_blood_pressure'] = pd.to_numeric(df['resting_blood_pressure'], errors='coerce')
    df['resting_blood_pressure'] = df['resting_blood_pressure'].fillna(df['resting_blood_pressure'].median())
    return df


def clean_st_depression(df):
    df['st_depression'] = df['st_depression'].astype(str).str.replace(',', '.')
    df['st_depression'] = pd.to_numeric(df['st_depression'], errors='coerce')
    df['st_depression'] = df['st_depression'].fillna(df['st_depression'].median())
    return df


def clean_thalassemia(df):
    most_frequent_thal = df['thalassemia'].mode()[0]
    df['thalassemia'] = df['thalassemia'].fillna(most_frequent_thal)
    df['thalassemia'] = pd.to_numeric(df['thalassemia'], errors='coerce')
    return df


def clean_data(df):
    df = drop_missing_rows(df)
    df = replace_question_marks(df)
    df = clean_age(df)
    df = remove_age_outliers(df)
    df = clean_gender(df)
    df = clean_cholesterol(df)
    df = clean_resting_blood_pressure(df)
    df = clean_st_depression(df)
    df = clean_thalassemia(df)
    return df

In [ ]:
# Visualization helpers
def plot_key_relationships(df):
    # Create a 2x2 grid of plots showing the strongest relationships
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Heart disease rate by chest pain type
    sns.barplot(data=df, x='chest_pain_type', y=TARGET, ax=axes[0, 0], palette='viridis')
    axes[0, 0].set_title('Heart Disease Rate by Chest Pain Type')
    axes[0, 0].set_ylabel('Heart Disease Probability')

    # 2. Strong negative relationship between age and max heart rate
    sns.regplot(data=df, x='age', y='max_heart_rate', ax=axes[0, 1],
                scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
    axes[0, 1].set_title('Age vs Max Heart Rate (Strong Negative Correlation)')

    # 3. Max heart rate distribution by heart disease target
    sns.boxplot(data=df, x=TARGET, y='max_heart_rate', ax=axes[1, 0], palette='Set2')
    axes[1, 0].set_title('Max Heart Rate by Heart Disease Target')

    # 4. Heart disease rate by number of major vessels
    sns.barplot(data=df, x='num_major_vessels', y=TARGET, ax=axes[1, 1], palette='mako')
    axes[1, 1].set_title('Heart Disease Rate by Number of Major Vessels')

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_test, y_pred):
    plt.figure(figsize=(6, 4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Disease', 'Disease'], yticklabels=['No Disease', 'Disease'])
    plt.title('Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

In [ ]:
# Modeling helpers:
# - encode categorical variables (e.g. gender) as numeric
# - split the data into 80% training and 20% testing
# - train a Random Forest model and evaluate its accuracy
def prepare_features(df):
    # Encode categorical variables as numeric
    df_encoded = pd.get_dummies(df, columns=['gender'], drop_first=True)

    # Define features (X) and target (y)
    X = df_encoded.drop(TARGET, axis=1)
    y = df_encoded[TARGET]
    return X, y


def split_data(X, y):
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


def train_model(X_train, y_train):
    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Accuracy Score: {accuracy:.2f} ({accuracy*100:.1f}%)")
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    plot_confusion_matrix(y_test, y_pred)

In [ ]:
# Load the raw data
df = load_data(DATA_PATH)
df

In [ ]:
# Check column data types
df.dtypes

In [ ]:
# Check the raw data structure and missing values
df.info()

In [ ]:
# Apply the full cleaning pipeline
df = clean_data(df)

In [ ]:
# Verify the cleaned data structure
df.info()

In [ ]:
# Inspect value distribution of every column after cleaning
show_value_counts(df)

In [ ]:
# Correlation matrix of numeric columns
df.corr(numeric_only=True)

In [ ]:
# Plot the strongest relationships
plot_key_relationships(df)

In [ ]:
# Prepare features, split data, train the model and evaluate it
X, y = prepare_features(df)
X_train, X_test, y_train, y_test = split_data(X, y)

model = train_model(X_train, y_train)
evaluate_model(model, X_test, y_test)